# CSV Serialization

eXist-db supports `method="csv"` as a serialization output method, producing [RFC 4180](https://www.rfc-editor.org/rfc/rfc4180)-compliant CSV from XQuery data structures. This is modeled on [BaseX's CSV support](https://docs.basex.org/13/CSV_Functions) and complements the XQuery 4.0 CSV functions.

## Array of Arrays

The most direct input: each inner array becomes a row, each member becomes a field.

In [ ]:
[
    ["Name", "Email", "Score"],
    ["Alice", "alice@example.com", 95],
    ["Bob", "bob@example.com", 87],
    ["Charlie", "charlie@example.com", 92]
]

By default, all fields are quoted. Use `csv.quotes=no` for minimal quoting — only fields containing commas, quotes, or newlines get quoted:

In [ ]:
serialize([
    ["Name", "Email", "Score"],
    ["Alice", "alice@example.com", 95],
    ["Bob", "bob@example.com", 87]
], map {
    "method": "csv",
    "csv.quotes": false()
})

## Sequence of Maps

When the input is a sequence of maps, keys become column names. Set `csv.header=yes` to output a header row with sorted key names:

In [ ]:
serialize((
    map { "name": "Alice",   "dept": "Engineering", "salary": 95000 },
    map { "name": "Bob",     "dept": "Marketing",   "salary": 82000 },
    map { "name": "Charlie", "dept": "Engineering", "salary": 91000 }
), map {
    "method": "csv",
    "csv.header": true()
})

This produces:

```
"dept","name","salary"
"Engineering","Alice","95000"
"Marketing","Bob","82000"
"Engineering","Charlie","91000"
```

## XML Tables

XML data in a record/field structure serializes directly:

In [ ]:
<csv>
    <record>
        <name>Alice</name>
        <age>30</age>
        <city>New York</city>
    </record>
    <record>
        <name>Bob</name>
        <age>25</age>
        <city>Portland</city>
    </record>
</csv>

## Custom Delimiters

### Tab-Separated Values (TSV)

In [ ]:
serialize([
    ["Product", "Price", "Qty"],
    ["Widget", "9.99", "100"],
    ["Gadget", "24.99", "50"]
], map {
    "method": "csv",
    "csv.field-delimiter": "&#9;",
    "csv.quotes": false()
})

### Pipe-Delimited

In [ ]:
serialize([
    ["ID", "Name", "Status"],
    ["1", "Alpha", "active"],
    ["2", "Beta|v2", "inactive"]
], map {
    "method": "csv",
    "csv.field-delimiter": "|",
    "csv.quotes": false()
})

Note how `"Beta|v2"` gets automatically quoted because it contains the delimiter.

## Quoting and Escaping

RFC 4180 requires doubling quote characters inside quoted fields:

In [ ]:
[
    ["Title", "Author", "Note"],
    ["The ""Great"" Gatsby", "Fitzgerald", "Classic"],
    ["It's a Wonderful Life", "Capra", "Contains apostrophe, no quoting needed"]
]

## Parameters Reference

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `csv.field-delimiter` | string | `,` | Character separating fields |
| `csv.row-delimiter` | string | `\n` | Line ending between rows |
| `csv.quote-character` | string | `"` | Character for quoting fields |
| `csv.header` | boolean | `false` | Output header row (for map input) |
| `csv.quotes` | boolean | `true` | `true` = always quote, `false` = quote only when needed |

## Real-World: Database Export

Query a collection and export as CSV for spreadsheet import:

In [ ]:
(: Assuming /db/people contains person documents :)
let $people := collection("/db/people")//person
return serialize(
    for $p in $people
    return map {
        "name": $p/name/string(),
        "email": $p/email/string(),
        "role": $p/role/string()
    },
    map { "method": "csv", "csv.header": true() }
)

## Real-World: RESTXQ CSV Endpoint

Create a downloadable CSV endpoint:

In [ ]:
(: In a RESTXQ module :)
(:
declare
    %rest:GET
    %rest:path("/api/report.csv")
    %output:method("csv")
    %output:media-type("text/csv")
function local:csv-report() {
    for $order in collection("/db/orders")//order
    return map {
        "id": $order/@id/string(),
        "customer": $order/customer/string(),
        "total": $order/total/string(),
        "date": $order/date/string()
    }
};
:)
"(See code comment above for RESTXQ pattern)"